In [0]:
%sql
USE CATALOG retail_dwh;
USE SCHEMA gold;

In [0]:
%sql
-- Highest revenue month

SELECT DATE_FORMAT(TxnDate,'yyyy-MM') AS Month,
       ROUND(SUM(Amount),2) AS Total_Revenue
FROM FactSales
GROUP BY Month
ORDER BY Total_Revenue DESC
LIMIT 1;

In [0]:
%sql
-- Monthly revenue trend

SELECT DATE_FORMAT(TxnDate,'yyyy-MM') AS Month,
       COUNT(TransactionID) AS Total_Transactions,
       ROUND(SUM(Amount),2) AS Revenue
FROM FactSales
GROUP BY Month
ORDER BY Month;

In [0]:
%sql
-- Top customers by spending

SELECT c.CustomerID,
       c.CustomerName,
       c.City,
       ROUND(SUM(f.Amount),2) AS Total_Spent
FROM FactSales f
JOIN DimCustomer c
ON f.CustomerSK = c.CustomerSK
WHERE c.IsActive = 1
GROUP BY c.CustomerID, c.CustomerName, c.City
ORDER BY Total_Spent DESC
LIMIT 10;

In [0]:
%sql
-- Top revenue generating products

SELECT p.ProductID,
       p.ProductName,
       p.Category,
       ROUND(SUM(f.Amount),2) AS Revenue
FROM FactSales f
JOIN DimProduct p
ON f.ProductSK = p.ProductSK
GROUP BY p.ProductID, p.ProductName, p.Category
ORDER BY Revenue DESC
LIMIT 10;

In [0]:
%sql
-- Best selling product categories

SELECT p.Category,
       SUM(f.Quantity) AS Units_Sold,
       ROUND(SUM(f.Amount),2) AS Revenue
FROM FactSales f
JOIN DimProduct p
ON f.ProductSK = p.ProductSK
GROUP BY p.Category
ORDER BY Revenue DESC;

In [0]:
%sql
-- Revenue by region

SELECT s.Region,
       COUNT(f.TransactionID) AS Transactions,
       ROUND(SUM(f.Amount),2) AS Revenue
FROM FactSales f
JOIN DimStore s
ON f.StoreSK = s.StoreSK
GROUP BY s.Region
ORDER BY Revenue DESC;

In [0]:
%sql
-- Store performance analysis

SELECT s.StoreName,
       s.Region,
       ROUND(SUM(f.Amount),2) AS Revenue,
       SUM(f.Quantity) AS Units_Sold
FROM FactSales f
JOIN DimStore s
ON f.StoreSK = s.StoreSK
GROUP BY s.StoreName, s.Region
ORDER BY Revenue DESC;

In [0]:
%sql
-- Average order value

SELECT ROUND(AVG(Amount),2) AS Avg_Order_Value
FROM FactSales;

In [0]:
%sql
-- Products never sold

SELECT p.ProductID,
       p.ProductName,
       p.Category
FROM DimProduct p
LEFT JOIN FactSales f
ON p.ProductSK = f.ProductSK
WHERE f.ProductSK IS NULL;

In [0]:
%sql
-- Customer history tracking using SCD Type 2

SELECT CustomerID,
       CustomerName,
       City,
       Address,
       StartDate,
       EndDate,
       CASE
            WHEN IsActive = 1 THEN 'Current'
            ELSE 'Historical'
       END AS Record_Status
FROM DimCustomer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM DimCustomer
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate;